## Imports and Constants

In [28]:
# Section 1 Imports
# To harvest the current date
import openactive as oa
from datetime import date
# Requests and retry for RPDE feeds
import requests
from urllib3.util import Retry
# Import json for formatting
import json
import pandas as pd
# import beautiful soup for web scraping links
from bs4 import BeautifulSoup

# Section 2 Imports
import json
from urllib.parse import urljoin
import time
import numpy as np
import io
import contextlib
import re

# Get current date to track when the dataset is downloaded
HARVEST_DATE = date.today().isoformat()

# A function for printing out formatted output as suggested by the OA package github
def printer(arg):
    print(json.dumps(arg, indent=4))

## SECTION 1: OpenActive website feeds to DF 
### METHOD 1: Get the table of all providers and feeds off of OpenActive's website

In [3]:
url = "https://status.openactive.io/"

# Get webpage
response = requests.get(url, timeout=30)
response.raise_for_status()

# Parse HTML
soup = BeautifulSoup(response.text, "html.parser")

# Find tables
tables = soup.find_all("table")

### Load the table into a Pandas DataFrame

In [43]:
# instantiate the data, each row of the table will be appended to this
data = []

# go through the first table
for table in tables[:1]:
    # Skip tables without headers
    headers = [th.get_text(strip=True) for th in table.find_all("th")]
    
    # Get the index for the column which "Provider" and "Feed Status" is in
    provider_index = headers.index("Provider")
    feed_index = headers.index("Feed status")
    
    # Loop through each row of the table
    for row in table.find_all("tr"):
        # For each row, get all columns
        cells = row.find_all("td")
        
        if len(cells) <= max(provider_index, feed_index):
            continue
        
        # Get the <a> marker that includes both href and anchor text
        provider_cell = cells[provider_index].find_all("a")[1]
        # Get the URL from the href
        provider_url = provider_cell.get("href")
        # Get the name through the anchor text 
        provider_name = provider_cell.text
        
        # Instantiate the "Feed status" column as feed_cell
        feed_cell = cells[feed_index]
        
        # Set the following feeds as None
        facilityUse = None
        slot = None
        sessionSeries = None
        scheduledSession = None
        event = None
        unnamedFeed = None
        
        # Loop through each <a> in feed_cell, 
        # if there is a valid link update the feeds above.
        # Otherwise, None will represent lack of feed
        for a in feed_cell.find_all("a"):
            if a.text == "FacilityUse":
                facilityUse = a.get("href")
            elif a.text == "Slot":
                slot = a.get("href")
            elif a.text == "SessionSeries":
                sessionSeries = a.get("href")
            elif a.text == "ScheduledSession":
                scheduledSession = a.get("href")
            elif a.text == "Event":
                event = a.get("href")
            elif a.text == "Unnamed Feed":
                unnamedFeed = a.get("href")
        
        # Append all the data scraped from the above code into the data list 
        data.append({
            "provider": provider_name,
            "provider_url": provider_url,
            "facilityUse": facilityUse,
            "slot": slot,
            "sessionSeries": sessionSeries,
            "scheduledSession": scheduledSession,
            "event": event,
            "unnamedFeed": unnamedFeed
        })

# Create DataFrame from the final data list after running the code above
method1_df = pd.DataFrame(data)

### Preliminary inspection of the loaded DF

In [44]:
# display the first few rows of the df
method1_df.head()

,provider,provider_url,facilityUse,slot,sessionSeries,scheduledSession,event,unnamedFeed
0,100% TO THE TOP CIC,https://topcic.bookteq.com/api/open-active/,https://topcic.bookteq.com/api/open-active/fac...,https://topcic.bookteq.com/api/open-active/slots,NaN,NaN,NaN,NaN
1,Actihire,https://actihire.bookteq.com/api/open-active/,https://actihire.bookteq.com/api/open-active/f...,https://actihire.bookteq.com/api/open-active/s...,NaN,NaN,NaN,NaN
2,Active Hartlepool,https://activehartlepool.gs-signature.cloud/Op...,https://opendata.leisurecloud.live/api/feeds/H...,https://opendata.leisurecloud.live/api/feeds/H...,https://opendata.leisurecloud.live/api/feeds/H...,https://opendata.leisurecloud.live/api/feeds/H...,NaN,NaN
3,Active Leeds,https://activeleeds-oa.leisurecloud.net/OpenAc...,https://opendata.leisurecloud.live/api/feeds/A...,https://opendata.leisurecloud.live/api/feeds/A...,https://opendata.leisurecloud.live/api/feeds/A...,https://opendata.leisurecloud.live/api/feeds/A...,NaN,NaN
4,Active Luton,https://activeluton-openactive.legendonlineser...,https://activeluton-openactive.legendonlineser...,https://activeluton-openactive.legendonlineser...,https://activeluton-openactive.legendonlineser...,NaN,NaN,NaN


In [45]:
method1_df.describe()

,provider,provider_url,facilityUse,slot,sessionSeries,scheduledSession,event,unnamedFeed
count,174,174,148,147,71,38,8,9
unique,171,174,147,146,70,37,8,9
top,Chelmsford City Sports,https://topcic.bookteq.com/api/open-active/,https://opendata.leisurecloud.live/api/feeds/C...,https://opendata.leisurecloud.live/api/feeds/C...,https://opendata.leisurecloud.live/api/feeds/C...,https://opendata.leisurecloud.live/api/feeds/C...,https://bookwhen.com/api/openactive/events,http://api.letsride.co.uk/public/v1/rides
freq,2,1,2,2,2,2,1,1


In [42]:
printer(feeds)

{
    "https://activehartlepool.gs-signature.cloud/OpenActive/": [
        {
            "type": "CourseInstance",
            "url": "https://opendata.leisurecloud.live/api/feeds/HartlepoolBoroughCouncil-live-course-instance",
            "dataset_name": "Active Hartlepool Sessions and Facilities",
            "dataset_url": "https://activehartlepool.gs-signature.cloud/OpenActive/",
            "discussion_url": "",
            "license_url": "https://creativecommons.org/licenses/by/4.0/",
            "logo_url": "https://res.cloudinary.com/gladstone/image/upload/HartlepoolBoroughCouncil-live/ztb0fzrqpsadn1ikfp7b",
            "publisher_name": "Active Hartlepool"
        },
        {
            "type": "SessionSeries",
            "url": "https://opendata.leisurecloud.live/api/feeds/HartlepoolBoroughCouncil-live-session-series",
            "dataset_name": "Active Hartlepool Sessions and Facilities",
            "dataset_url": "https://activehartlepool.gs-signature.cloud/OpenActive/

### METHOD 2: Using the OpenActive Package
Store all successful and failed feed calls into method2_feeds_df

In [30]:
# Capture printed output from oa.get_feeds()
output = io.StringIO()

with contextlib.redirect_stdout(output), contextlib.redirect_stderr(output):
    feeds = oa.get_feeds()

# Store all printed warnings/errors
errors = output.getvalue()

# Extract failed URLs from error messages
failed_urls = re.findall(
    r"ERROR: Can't get dataset: (.+)",
    errors
)

# Create table rows
data = []

# Add successful feeds
for provider_url, feed_data in feeds.items():
    data.append({
        "feed_url": provider_url,
        "status": "Success"
    })

# Add failed feeds
for url in failed_urls:
    data.append({
        "feed_url": url,
        "status": "Failed",
    })

# Convert to DataFrame
method2_feeds_df = pd.DataFrame(data)

method2_feeds_df.head()

,feed_url,status
0,https://activehartlepool.gs-signature.cloud/Op...,Success
1,https://activeleeds-oa.leisurecloud.net/OpenAc...,Success
2,https://bccleisure.gs-signature.cloud/OpenActive/,Success
3,https://bewellwigan.gs-signature.cloud/OpenAct...,Success
4,https://brimhamsactive.gs-signature.cloud/Open...,Success


Go through each row of method2_feeds_df, depending on whether it succeeded or failed in getting a url, store the feeds gained into method2_df.

In [47]:
# instantiate where each row of data will be, at the end will convert to df
data = []

# iterate through each row in the method2_feeds_df
for _, row in method2_feeds_df.iterrows():
    # handle logic for valid oa.get_feeds() outputs
    if row["status"] == "Success":
        # instantiate the following variables as None
        provider_name = None
        provider_url = None
        facilityUse = None
        slot = None
        sessionSeries = None
        scheduledSession = None
        event = None
        unnamedFeed = np.nan # changed to np.nan because for some reason the table would save "None" rather than NaN
        
        # get the current provider feeds
        current_provider_feeds = feeds[row["feed_url"]]
        # check the type of the current provider feed and update the None variables above if available
        for feed in current_provider_feeds:
            if feed["type"] == "FacilityUse":
                facilityUse = feed["url"]
            elif feed["type"] == "Slot":
                slot = feed["url"]
            elif feed["type"] == "SessionSeries":
                sessionSeries = feed["url"]
            elif feed["type"] == "ScheduledSession":
                scheduledSession = feed["url"]
            elif feed["type"] == "Event":
                event = feed["url"]
            
            # because the oa schema may or may not have "publisher_name", we use .get
            provider_name = feed.get("publisher_name", None)
            if provider_name == "": # Some publishers have provider_name set to "" which != None or NaN so we have to change it
                provider_name = None
            provider_url = feed.get("dataset_url", None) # get the provider_url 
        
        # append each of the variable as a row to the data list
        data.append({
            "provider": provider_name,
            "provider_url": provider_url,
            "facilityUse": facilityUse,
            "slot": slot,
            "sessionSeries": sessionSeries,
            "scheduledSession": scheduledSession,
            "event": event,
            "unnamedFeed": unnamedFeed
        })
    
    # handle logic for failed oa.get_feeds() outputs
    elif row["status"] == "Failed":
        # has to set all to None except provider_url, as errors only pass back a printed error message and url with no other information
        data.append({
            "provider": None,
            "provider_url": row["feed_url"],
            "facilityUse": None,
            "slot": None,
            "sessionSeries": None,
            "scheduledSession": None,
            "event": None,
            "unnamedFeed": np.nan # changed to np.nan because for some reason the table would save "None" rather than NaN
        })

# convert to DF
method2_df = pd.DataFrame(data)

# preview of DF head
method2_df.head()

,provider,provider_url,facilityUse,slot,sessionSeries,scheduledSession,event,unnamedFeed
0,Active Hartlepool,https://activehartlepool.gs-signature.cloud/Op...,https://opendata.leisurecloud.live/api/feeds/H...,https://opendata.leisurecloud.live/api/feeds/H...,https://opendata.leisurecloud.live/api/feeds/H...,https://opendata.leisurecloud.live/api/feeds/H...,NaN,NaN
1,Active Leeds,https://activeleeds-oa.leisurecloud.net/OpenAc...,https://opendata.leisurecloud.live/api/feeds/A...,https://opendata.leisurecloud.live/api/feeds/A...,https://opendata.leisurecloud.live/api/feeds/A...,https://opendata.leisurecloud.live/api/feeds/A...,NaN,NaN
2,Birmingham City Council,https://bccleisure.gs-signature.cloud/OpenActive/,https://opendata.leisurecloud.live/api/feeds/B...,https://opendata.leisurecloud.live/api/feeds/B...,https://opendata.leisurecloud.live/api/feeds/B...,https://opendata.leisurecloud.live/api/feeds/B...,NaN,NaN
3,Wigan Leisure and Culture Trust,https://bewellwigan.gs-signature.cloud/OpenAct...,https://opendata.leisurecloud.live/api/feeds/W...,https://opendata.leisurecloud.live/api/feeds/W...,https://opendata.leisurecloud.live/api/feeds/W...,https://opendata.leisurecloud.live/api/feeds/W...,NaN,NaN
4,Brimhams Active,https://brimhamsactive.gs-signature.cloud/Open...,https://opendata.leisurecloud.live/api/feeds/B...,https://opendata.leisurecloud.live/api/feeds/B...,https://opendata.leisurecloud.live/api/feeds/B...,https://opendata.leisurecloud.live/api/feeds/B...,NaN,NaN


## Section 1 Outputs Evaluation

In [ ]:
print(
    "Method 1: Evaluation\n" 
    
    f"Number of Providers: {len(method1_df)}\n"
    
    f"Number of Missing Provider Names: {len(method1_df[method1_df["provider"].isna()])}\n" 
        
    f"Number of Providers with no URLs: {len(method1_df[
    method1_df[
        ["facilityUse", "slot", "sessionSeries", "scheduledSession", "event", "unnamedFeed"]
    ].isna().all(axis=1)])}\n\n" 

    "Method 2: Evaluation\n" 
    f"Number of Providers: {len(method2_df)}\n"
        
    f"Number of Missing Provider Names: {len(method2_df[method2_df["provider"].isna()])}\n"
    
    f"Number of Providers with no URLS: {len(method2_df[
    method2_df[
        ["facilityUse", "slot", "sessionSeries", "scheduledSession", "event", "unnamedFeed"]
    ].isna().all(axis=1)])}"
    )


# Get the providers that aren't in each others tables


Method 1: Evaluation
Number of Providers: 174
Number of Missing Provider Names: 0
Number of Providers with no URLs: 9

Method 2: Evaluation
Number of Providers: 173
Number of Missing Provider Names: 23
Number of Providers with no URLS: 14


## SECTION 2:

In [ ]:
def harvest_feed(start_url, session, max_pages=20000, sleep=0.25, log_every=25):
    
    # Page through an OpenActive RPDE feed, following 'next' to the live edge.
    # Returns (current_state, stats): current_state maps id -> the full RPDE item.
    current = {}
    seen_updated = seen_deleted = pages = 0
    url, last_url = start_url, None
    while url and url != last_url and pages < max_pages:
        resp = session.get(url, timeout=60)
        resp.raise_for_status()
        payload = resp.json()
        items = payload.get('items', [])
        if not items:
            break                      # empty page = caught up to live
        for item in items:
            item_id = item.get('id')
            if item_id is None:
                continue
            state = item.get('state')
            if state == 'updated':
                current[item_id] = item
                seen_updated += 1
            elif state == 'deleted':
                current.pop(item_id, None)
                seen_deleted += 1
        nxt = payload.get('next')
        if nxt:
            nxt = urljoin(url, nxt)    # handle relative next URLs
        last_url, url = url, nxt
        pages += 1
        if log_every and pages % log_every == 0:
            print(f'  page {pages:>5} | live: {len(current):>6} | updated seen: {seen_updated:>6} | deleted seen: {seen_deleted:>6}')
        if sleep:
            time.sleep(sleep)
    stats = {'pages': pages, 'seen_updated': seen_updated, 'seen_deleted': seen_deleted, 'live_count': len(current)}
    return current, stats

harvest_feed("https://actihire.bookteq.com/api/open-active/facility-uses", session)

({'53b2c417-98a0-4a3a-a323-508b5c6c5104': {'state': 'updated',
   'kind': 'IndividualFacilityUse',
   'id': '53b2c417-98a0-4a3a-a323-508b5c6c5104',
   'modified': 1728405430591807,
   'data': {'@type': 'IndividualFacilityUse',
    '@context': ['https://openactive.io/', 'https://openactive.io/ns-beta'],
    '@id': 'https://actihire.bookteq.com/api/open-active/46bab773-d720-4a38-8638-c7a9f0c0b96f/facility-uses/53b2c417-98a0-4a3a-a323-508b5c6c5104',
    'identifier': '53b2c417-98a0-4a3a-a323-508b5c6c5104',
    'name': 'Site Viewing',
    'url': 'https://widget.bookteq.com/actihire/book-online/176f7e5b-bba1-4383-8bae-275f6f122537',
    'facilityType': [{'@type': 'Concept',
      '@id': 'https://openactive.io/facility-types#da364f9b-8bb2-490e-9e2f-1068790b9e35',
      'inScheme': 'https://openactive.io/facility-type',
      'prefLabel': 'Sports Hall'}],
    'location': {'@type': 'Place',
     '@id': 'https://actihire.bookteq.com/api/open-active/place/46bab773-d720-4a38-8638-c7a9f0c0b96f',
 